# Extra 02: Extra tuning experiments (validation only)

**The test set is not loaded

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

SEED = 12
CACHE = Path("../cache"); EXP = Path("../experiments"); EXP.mkdir(exist_ok=True)
imgs = np.load(CACHE / "images_uint8.npz"); feats = np.load(CACHE / "features.npz")
X_train, X_val = imgs["X_train"], imgs["X_val"]
y_train, y_val = feats["y_train"], feats["y_val"]

def to_raw(X):
    return X.reshape(len(X), -1).astype(np.float32) / 255.0

rows = []
def add(exp, model, feat, setting, p):
    rows.append({"experiment": exp, "model": model, "features": feat, "setting": setting,
                 "val_acc": accuracy_score(y_val, p), "val_macro_f1": f1_score(y_val, p, average="macro")})

def run_knn(exp, feat, Xtr, Xva, ks=(1, 3, 5, 9, 15), metrics=("euclidean",)):
    for metric in metrics:
        for k in ks:
            m = KNeighborsClassifier(n_neighbors=k, metric=metric, algorithm="brute").fit(Xtr, y_train)
            add(exp, "KNN", feat, f"k={k}, metric={metric}", m.predict(Xva))

def run_lr(exp, feat, Xtr, Xva, Cs):
    sc = StandardScaler().fit(Xtr); Ztr, Zva = sc.transform(Xtr), sc.transform(Xva)
    for C in Cs:
        m = LogisticRegression(C=C, max_iter=2000, random_state=SEED).fit(Ztr, y_train)
        add(exp, "LogReg", feat, f"C={C}", m.predict(Zva))

ref = pd.read_csv(EXP / "best_per_model_part_b.csv")
print("current best on validation (notebook 03):")
print(ref[["model", "features", "param", "val_acc"]].to_string(index=False))

current best on validation (notebook 03):
 model features          param  val_acc
   KNN     hist k=1, w=uniform   0.6792
   KNN      pca k=1, w=uniform   0.5427
   KNN      raw k=1, w=uniform   0.5256
LogReg     hist          C=0.1   0.6109
LogReg      pca         C=0.01   0.4061
LogReg      raw       C=0.0001   0.4300


## B1. Logistic Regression: wider C grid on raw pixels
The best C was the smallest one tried (0.0001), so the optimum may be lower.

In [2]:
Rtr, Rva = to_raw(X_train), to_raw(X_val)
run_lr("B1 wider C", "raw", Rtr, Rva, [1e-6, 1e-5, 1e-4])
pd.DataFrame(rows).round(4)

,experiment,model,features,setting,val_acc,val_macro_f1
0,B1 wider C,LogReg,raw,C=1e-06,0.3242,0.3192
1,B1 wider C,LogReg,raw,C=1e-05,0.3618,0.3609
2,B1 wider C,LogReg,raw,C=0.0001,0.4300,0.4295


## B2. KNN: distance metric
`euclidean` was the default. `manhattan` is less sensitive to a few large differences, `cosine` ignores the overall brightness of the vector.

In [3]:
run_knn("B2 metric", "hist", feats["H_train"], feats["H_val"], metrics=("euclidean", "manhattan", "cosine"))
run_knn("B2 metric", "pca",  feats["P_train"], feats["P_val"], metrics=("euclidean", "manhattan", "cosine"))
run_knn("B2 metric", "raw",  Rtr, Rva, ks=(1, 5), metrics=("euclidean", "manhattan", "cosine"))
d = pd.DataFrame(rows); d[d.experiment == "B2 metric"].round(4)

,experiment,model,features,setting,val_acc,val_macro_f1
3,B2 metric,KNN,hist,"k=1, metric=euclidean",0.6792,0.6800
4,B2 metric,KNN,hist,"k=3, metric=euclidean",0.6280,0.6287
5,B2 metric,KNN,hist,"k=5, metric=euclidean",0.5939,0.5933
6,B2 metric,KNN,hist,"k=9, metric=euclidean",0.5734,0.5707
7,B2 metric,KNN,hist,"k=15, metric=euclidean",0.4744,0.4630
8,B2 metric,KNN,hist,"k=1, metric=manhattan",0.7406,0.7404
9,B2 metric,KNN,hist,"k=3, metric=manhattan",0.6928,0.6946
10,B2 metric,KNN,hist,"k=5, metric=manhattan",0.6416,0.6409
11,B2 metric,KNN,hist,"k=9, metric=manhattan",0.6041,0.6009
12,B2 metric,KNN,hist,"k=15, metric=manhattan",0.5358,0.5350


## C1. Color-histogram variants
Different number of bins per channel in RGB, and HSV (hue / saturation / value), which separates color from brightness.

In [4]:
def joint_hist(q, bins):
    
    n = len(q); nb = bins[0] * bins[1] * bins[2]
    idx = (q[..., 0] * bins[1] * bins[2] + q[..., 1] * bins[2] + q[..., 2]).reshape(n, -1)
    idx = idx + np.arange(n)[:, None] * nb
    H = np.bincount(idx.ravel(), minlength=n * nb).reshape(n, nb).astype(np.float32)
    return H / H.sum(axis=1, keepdims=True)

def hist_rgb(X, b):
    return joint_hist((X // (256 // b)).astype(np.int64), (b, b, b))

def hist_hsv(X, bins):
    out = []
    for s in range(0, len(X), 200):
        hsv = rgb_to_hsv(X[s:s + 200].astype(np.float32) / 255.0)
        q = np.stack([np.minimum((hsv[..., c] * bins[c]).astype(np.int64), bins[c] - 1) for c in range(3)], axis=-1)
        out.append(joint_hist(q, bins))
    return np.vstack(out)

variants = {
    "rgb 4 bins (64)":     lambda X: hist_rgb(X, 4),
    "rgb 16 bins (4096)":  lambda X: hist_rgb(X, 16),
    "hsv 8x4x4 (128)":     lambda X: hist_hsv(X, (8, 4, 4)),
    "hsv 16x8x8 (1024)":   lambda X: hist_hsv(X, (16, 8, 8)),
}
HV = {}
for name, fn in variants.items():
    HV[name] = (fn(X_train), fn(X_val))
    Htr, Hva = HV[name]
    run_knn("C1 hist variant", name, Htr, Hva, ks=(1, 3, 5), metrics=("euclidean", "manhattan"))
    run_lr("C1 hist variant", name, Htr, Hva, [0.01, 0.1, 1])
    print("done:", name)
d = pd.DataFrame(rows); d[d.experiment == "C1 hist variant"].round(4)

done: rgb 4 bins (64)
done: rgb 16 bins (4096)
done: hsv 8x4x4 (128)
done: hsv 16x8x8 (1024)


,experiment,model,features,setting,val_acc,val_macro_f1
39,C1 hist variant,KNN,rgb 4 bins (64),"k=1, metric=euclidean",0.6314,0.6326
40,C1 hist variant,KNN,rgb 4 bins (64),"k=3, metric=euclidean",0.5529,0.5523
41,C1 hist variant,KNN,rgb 4 bins (64),"k=5, metric=euclidean",0.5188,0.5143
42,C1 hist variant,KNN,rgb 4 bins (64),"k=1, metric=manhattan",0.6894,0.6898
43,C1 hist variant,KNN,rgb 4 bins (64),"k=3, metric=manhattan",0.6246,0.6245
44,C1 hist variant,KNN,rgb 4 bins (64),"k=5, metric=manhattan",0.5529,0.5492
45,C1 hist variant,LogReg,rgb 4 bins (64),C=0.01,0.3823,0.3803
46,C1 hist variant,LogReg,rgb 4 bins (64),C=0.1,0.4266,0.4251
47,C1 hist variant,LogReg,rgb 4 bins (64),C=1,0.4096,0.4087
48,C1 hist variant,KNN,rgb 16 bins (4096),"k=1, metric=euclidean",0.6758,0.6763


## C2. Smaller images (32 x 32)
Average-pooling the cached 64 x 64 images. (64 -> 128 would need reloading the original files, so it is not covered here.)

In [5]:
def pool32(X):
    n = len(X)
    return X.reshape(n, 32, 2, 32, 2, 3).mean(axis=(2, 4)).astype(np.float32) / 255.0

S_tr, S_va = pool32(X_train).reshape(len(X_train), -1), pool32(X_val).reshape(len(X_val), -1)
print("raw 32x32 features:", S_tr.shape[1])
run_knn("C2 size 32", "raw32", S_tr, S_va, ks=(1, 3, 5), metrics=("euclidean", "manhattan"))
run_lr("C2 size 32", "raw32", S_tr, S_va, [1e-5, 1e-4, 1e-3, 1e-2])
d = pd.DataFrame(rows); d[d.experiment == "C2 size 32"].round(4)

raw 32x32 features: 3072


,experiment,model,features,setting,val_acc,val_macro_f1
75,C2 size 32,KNN,raw32,"k=1, metric=euclidean",0.5392,0.5368
76,C2 size 32,KNN,raw32,"k=3, metric=euclidean",0.4710,0.4645
77,C2 size 32,KNN,raw32,"k=5, metric=euclidean",0.4505,0.4425
78,C2 size 32,KNN,raw32,"k=1, metric=manhattan",0.6007,0.6014
79,C2 size 32,KNN,raw32,"k=3, metric=manhattan",0.5222,0.5180
80,C2 size 32,KNN,raw32,"k=5, metric=manhattan",0.4676,0.4573
81,C2 size 32,LogReg,raw32,C=1e-05,0.3515,0.3524
82,C2 size 32,LogReg,raw32,C=0.0001,0.3823,0.3813
83,C2 size 32,LogReg,raw32,C=0.001,0.4403,0.4398
84,C2 size 32,LogReg,raw32,C=0.01,0.4027,0.4010


## C3. HOG (edges and shapes instead of colors)
Needs `scikit-image`. If the import fails, start Jupyter with `uv run --with scikit-image jupyter lab`
(this does not touch the shared `pyproject.toml`), or ask the team leader to add it for everyone.

In [6]:
try:
    from skimage.feature import hog
except ImportError:
    hog = None
    print("scikit-image is not installed: skip this section (see the note above).")

if hog is not None:
    def hog_features(X):
        gray = (X.astype(np.float32) @ np.array([0.299, 0.587, 0.114], dtype=np.float32)) / 255.0
        return np.stack([hog(g, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)) for g in gray]).astype(np.float32)

    G_tr, G_va = hog_features(X_train), hog_features(X_val)
    print("HOG features:", G_tr.shape[1])
    run_knn("C3 HOG", "hog", G_tr, G_va, ks=(1, 3, 5, 9), metrics=("euclidean", "manhattan"))
    run_lr("C3 HOG", "hog", G_tr, G_va, [0.0001, 0.001, 0.01, 0.1])
    Htr, Hva = feats["H_train"], feats["H_val"]
    run_lr("C3 HOG", "hist+hog", np.hstack([Htr, G_tr]), np.hstack([Hva, G_va]), [0.0001, 0.001, 0.01, 0.1])
    d = pd.DataFrame(rows); print(d[d.experiment == "C3 HOG"].round(4).to_string(index=False))

HOG features: 1764
experiment  model features               setting  val_acc  val_macro_f1
    C3 HOG    KNN      hog k=1, metric=euclidean   0.5973        0.5987
    C3 HOG    KNN      hog k=3, metric=euclidean   0.5461        0.5438
    C3 HOG    KNN      hog k=5, metric=euclidean   0.5290        0.5280
    C3 HOG    KNN      hog k=9, metric=euclidean   0.4710        0.4655
    C3 HOG    KNN      hog k=1, metric=manhattan   0.6314        0.6312
    C3 HOG    KNN      hog k=3, metric=manhattan   0.5734        0.5753
    C3 HOG    KNN      hog k=5, metric=manhattan   0.5734        0.5703
    C3 HOG    KNN      hog k=9, metric=manhattan   0.5358        0.5350
    C3 HOG LogReg      hog              C=0.0001   0.4300        0.4278
    C3 HOG LogReg      hog               C=0.001   0.4642        0.4609
    C3 HOG LogReg      hog                C=0.01   0.4744        0.4719
    C3 HOG LogReg      hog                 C=0.1   0.4505        0.4501
    C3 HOG LogReg hist+hog              C=0.0

## Summary: best of each experiment vs the current best

In [7]:
res = pd.DataFrame(rows)
res.to_csv(EXP / "extra_experiments_part_b.csv", index=False)

best_now = ref.groupby("model")["val_acc"].max()
print("current best  | LogReg:", round(best_now.get("LogReg", np.nan), 4), "| KNN:", round(best_now.get("KNN", np.nan), 4))
print("validation set size:", len(y_val), "-> margin of error about +-0.03")
print()
top = res.loc[res.groupby(["experiment", "model", "features"])["val_acc"].idxmax()]
print(top.sort_values("val_acc", ascending=False)[["experiment", "model", "features", "setting", "val_acc", "val_macro_f1"]].round(4).head(15).to_string(index=False))

current best  | LogReg: 0.6109 | KNN: 0.6792
validation set size: 293 -> margin of error about +-0.03

     experiment  model           features               setting  val_acc  val_macro_f1
C1 hist variant    KNN rgb 16 bins (4096) k=1, metric=manhattan   0.7986        0.7978
C1 hist variant    KNN  hsv 16x8x8 (1024) k=1, metric=manhattan   0.7884        0.7879
C1 hist variant    KNN    hsv 8x4x4 (128) k=1, metric=manhattan   0.7679        0.7669
      B2 metric    KNN               hist k=1, metric=manhattan   0.7406        0.7404
C1 hist variant LogReg rgb 16 bins (4096)                   C=1   0.7133        0.7129
C1 hist variant LogReg  hsv 16x8x8 (1024)                 C=0.1   0.6894        0.6900
C1 hist variant    KNN    rgb 4 bins (64) k=1, metric=manhattan   0.6894        0.6898
         C3 HOG    KNN                hog k=1, metric=manhattan   0.6314        0.6312
     C2 size 32    KNN              raw32 k=1, metric=manhattan   0.6007        0.6014
      B2 metric    KNN     